# Nigeria Premier Football League (NPFL) Club Info Analysis

## Objective


Scrape the NPFL Club Info table from Wikipedia, clean stadium capacities,
Export a CSV, and analyze top stadiums and locations.

This project explores Nigerian football club performance data to uncover trends, patterns, 
and strategic insights using Python.

Tools used:
- Python
- Pandas
- Matplotlib / Seaborn
- Jupyter Notebook

In [1]:
# explanation:
# Import standard libraries:
# - requests to fetch HTML pages
# - BeautifulSoup to parse HTML tables
# - pandas to clean, analyze, and export data


import requests
from bs4 import BeautifulSoup
import pandas as pd

In [8]:
# explanation:
# To scrape Wikipedia safely, i include a User-Agent header
# to prevent a 403 forbidden error.


url = "https://en.wikipedia.org/wiki/2023%E2%80%9324_Nigeria_Premier_Football_League"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)

# Check that page was fetched successfully (200 = OK)
response.status_code

200

In [9]:
# explanation:
# BeautifulSoup converts the raw HTML into a navigable structure.
# This allows us to locate the table containing club info.


soup = BeautifulSoup(response.text, "html.parser")

In [10]:
# explanation:
# Wikipedia tables usually have the class 'wikitable'.
# We find the first table, which contains club info.


table = soup.find("table", {"class": "wikitable"})

In [11]:
# explanation:
# Extract column names for the DataFrame.
# This gives us headers like: Team, Location, Kit Supplier, Shirt Sponsor, Stadium, Capacity

headers = [th.text.strip() for th in table.find_all("th")]
headers

['Team', 'Location', 'Kit Supplier', 'Shirt Sponsor', 'Stadium', 'Capacity']

In [12]:
# explanation:
# Loop through each row (<tr>) in the table, skip empty rows,
# and store cell text in a list.
rows = table.find_all("tr")[1:]  # skip header row
data = []

for row in rows:
    cells = row.find_all("td")
    if len(cells) == 0:
        continue
    row_data = [cell.text.strip() for cell in cells]
    data.append(row_data)

# Preview first 20 rows for sanity check
data[:20]

[['Abia Warriors',
  'Umuahia',
  'Owu',
  'Eunisell',
  'Umuahia Township StadiumEnyimba International Stadium[a]',
  '5,000'],
 ['Akwa United',
  'Uyo',
  'Owu',
  '1xBet',
  'Godswill Akpabio StadiumEket Stadium',
  '30,00018,000'],
 ['Bayelsa United', 'Yenagoa', 'GDF', '_', 'Samson Siasia Stadium', '5,000'],
 ['Bendel Insurance',
  'Benin City',
  'Fourteen',
  'Sterling Bank',
  'Samuel Ogbemudia Stadium',
  '12,000'],
 ['Doma United', 'Gombe', 'Cone Sport', '_', 'Pantami Stadium', '12,000'],
 ['Enugu Rangers',
  'Enugu',
  'Cone Sport',
  'Afrinvest,Shopurban, Senior Barman',
  'Nnamdi Azikiwe Stadium',
  '22,000'],
 ['Enyimba',
  'Aba',
  'KK',
  'Stake.com, United Nigeria Airlines',
  'Enyimba International Stadium',
  '16,000'],
 ['Gombe United', 'Gombe', 'Cone Sport', '_', 'Pantami Stadium', '12,000'],
 ['Heartland',
  'Owerri',
  '_',
  '_',
  'Dan Anyiam StadiumAwka City Stadium[b]',
  '10,000'],
 ['Kano Pillars', 'Kano', 'Play', 'Rambo', 'Sani Abacha Stadium', '16,000'],
 

In [15]:
# explanation:
# Convert scraped rows into a pandas DataFrame with proper headers.
# This makes the data easy to clean, analyze, and export.
df = pd.DataFrame(data, columns=headers)
df.head()

,Team,Location,Kit Supplier,Shirt Sponsor,Stadium,Capacity
0,Abia Warriors,Umuahia,Owu,Eunisell,Umuahia Township StadiumEnyimba International ...,"5,000"
1,Akwa United,Uyo,Owu,1xBet,Godswill Akpabio StadiumEket Stadium,"30,00018,000"
2,Bayelsa United,Yenagoa,GDF,_,Samson Siasia Stadium,"5,000"
3,Bendel Insurance,Benin City,Fourteen,Sterling Bank,Samuel Ogbemudia Stadium,"12,000"
4,Doma United,Gombe,Cone Sport,_,Pantami Stadium,"12,000"


In [20]:
# explanation:
# Some Capacity cells still have extra text or multiple numbers
# (like '30,00018,000'). We clean them by:
# 1. Converting everything to string first
# 2. Removing commas
# 3. Extracting only the first number
# 4. Converting to float so we can sort/analyze
# this was too complex, and was done with ai
import re

def clean_capacity(value):
    if pd.isna(value):
        return None
    value_str = str(value)              # ensure string
    value_str = value_str.replace(",", "")  # remove commas
    match = re.search(r"\d+", value_str)   # extract first numeric sequence
    if match:
        return float(match.group())
    return None

df["Capacity"] = df["Capacity"].apply(clean_capacity)

# check the first 10 rows
df[["Team", "Stadium", "Capacity"]].head()

,Team,Stadium,Capacity
0,Abia Warriors,Umuahia Township StadiumEnyimba International ...,5.000000e+03
1,Akwa United,Godswill Akpabio StadiumEket Stadium,3.000018e+09
2,Bayelsa United,Samson Siasia Stadium,5.000000e+03
3,Bendel Insurance,Samuel Ogbemudia Stadium,1.200000e+04
4,Doma United,Pantami Stadium,1.200000e+04


In [21]:
# explanation:
# The Capacity values are correct but displayed in scientific notation.
# Converting to integer makes them display as normal numbers.

df["Capacity"] = df["Capacity"].astype("Int64")  # nullable integer type
df[["Team", "Stadium", "Capacity"]].head(10)

,Team,Stadium,Capacity
0,Abia Warriors,Umuahia Township StadiumEnyimba International ...,5000
1,Akwa United,Godswill Akpabio StadiumEket Stadium,3000018000
2,Bayelsa United,Samson Siasia Stadium,5000
3,Bendel Insurance,Samuel Ogbemudia Stadium,12000
4,Doma United,Pantami Stadium,12000
5,Enugu Rangers,Nnamdi Azikiwe Stadium,22000
6,Enyimba,Enyimba International Stadium,16000
7,Gombe United,Pantami Stadium,12000
8,Heartland,Dan Anyiam StadiumAwka City Stadium[b],10000
9,Kano Pillars,Sani Abacha Stadium,16000


In [23]:
# explanation:
# I had to use my PC's full path to ensure the CSV is saved
# where I can easily access it.
# On my PC: C:\Users\USER\Documents\Python files


df.to_csv(r"C:\Users\USER\Documents\Python files\npfl_club_info.csv", index=False)

In [24]:
# explanation:
# Sort the clubs by stadium capacity to identify the largest stadiums in NPFL

top_stadiums = df.sort_values(by="Capacity", ascending=False).head(5)
top_stadiums[["Team", "Stadium", "Capacity"]]

,Team,Stadium,Capacity
1,Akwa United,Godswill Akpabio StadiumEket Stadium,3000018000
14,Plateau United,New Jos Stadium,60000
16,Rivers United,Adokiye Amiesimaka Stadium,40000
10,Katsina United,Muhammadu Dikko Stadium,35000
5,Enugu Rangers,Nnamdi Azikiwe Stadium,22000


## Business Insight Summary

Analyzing NPFL club infrastructure provides insights into stadium capacities,
geographical distribution of teams, and potential fan engagement.

The top stadiums in the league indicate where the largest crowds can attend games,
which is useful for sponsors, marketing campaigns, and league growth strategy.